# Rex Bot – Pipeline dữ liệu BAF → typed-decisions

Chuyển **Bank Account Fraud (NeurIPS 2022)** sang format `state` / `questions` / `gold`
(giống `LocalLLaMA/typed-decisions`), rồi chia **train / valid / test theo thời gian**.

**Chuẩn bị (1 lần):**
1. Bên phải notebook → **Input** → **+ Add Input** → tìm `Bank Account Fraud Dataset Suite (NeurIPS 2022)` (tác giả *sgpjesus*) → **Add**.
2. **Settings → Accelerator: None** (không cần GPU). Không cần bật Internet.
3. **Run All**. Bản đầy đủ chạy khoảng 5–15 phút.

**Kết quả** nằm ở `/kaggle/working/rexbot_baf/` (tab **Output** sau khi *Save Version*).

## 1. Cấu hình

In [ ]:
# ====== CHỈNH Ở ĐÂY ======
CSV_PATH      = None      # None = tự tìm Base.csv trong /kaggle/input
OUT_DIR       = "/kaggle/working/rexbot_baf"
NROWS         = None      # vd 200_000 để chạy thử nhanh; None = toàn bộ 1 triệu dòng

TEACHER_FOLDS = 5         # số fold cho teacher model (3 = nhanh hơn)
NEG_RATIO     = 9         # train: số đơn bình thường / 1 đơn gian lận
HARD_FRAC     = 0.3       # train: tỉ lệ đơn bình thường "khó" (teacher chấm điểm cao)
VALID_MONTH   = 6         # train = tháng < 6, valid = tháng 6, test = tháng > 6
VALID_N       = 20_000    # số dòng valid (giữ tỉ lệ fraud tự nhiên)
TEST_N        = 40_000    # số dòng test  (giữ tỉ lệ fraud tự nhiên)
N_ANNOTATORS  = 30        # số "annotator" bỏ phiếu tạo gold mềm
LABEL_ANCHOR  = 1.0       # tăng (vd 1.3) nếu quá nhiều ca fraud bị gán ALLOW (không nên > 1.5)
NOISE         = 0.4       # độ nhiễu của annotator: lớn hơn -> gold mềm hơn
PROTECT_LEGIT = True      # khách hợp lệ không bao giờ bị gán BLOCK (tối đa ESCALATE_HUMAN)
WRITE_ALL     = False     # True = ghi thêm baf_all.jsonl (trùng nội dung 3 file kia, ~400 MB)
SEED          = 42

In [ ]:
import glob, json, os, time, zipfile
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

def log(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)
def jdump(o): return json.dumps(o, ensure_ascii=False, separators=(",", ":"))

CSV_PATH = CSV_PATH or os.environ.get("REXBOT_CSV")
if CSV_PATH is None:
    all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    hits = [p for p in all_csv if os.path.basename(p).lower() == "base.csv"]
    if not hits:
        print("Không thấy Base.csv. Dataset đã gắn:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "không có")
        print("Các file CSV đang có:")
        for p in all_csv:
            print("  ", p)
        raise FileNotFoundError(
            "Gắn dataset 'Bank Account Fraud Dataset Suite (NeurIPS 2022)' của sgpjesus (Input > + Add Input), "
            "đợi vài giây rồi chạy lại; hoặc gán CSV_PATH = '<đường dẫn ở trên>' trong cell cấu hình.")
    CSV_PATH = hits[0]
os.makedirs(OUT_DIR, exist_ok=True)
log(f"Dùng file: {CSV_PATH}")

## 2. Hằng số và bộ câu hỏi chung của Rex Bot

In [ ]:
DECISIONS  = ["BLOCK", "ESCALATE_HUMAN", "TRIGGER_AI", "ALLOW"]
TRACE_OPTS = ["audit_log", "internet_cti", "syntax", "none"]
WORKFLOW   = "account_opening"

# BAF dùng số âm để mã hoá giá trị thiếu ở các cột này
NEG_MISSING = ["prev_address_months_count", "current_address_months_count",
               "bank_months_count", "session_length_in_minutes",
               "device_distinct_emails_8w", "intended_balcon_amount"]
PROTECTED_COLS = ["customer_age"]   # không đưa vào state/teacher (tránh chặn theo tuổi)
VEL_COLS = ["velocity_6h", "velocity_24h", "velocity_4w", "zip_count_4w",
            "bank_branch_count_8w", "date_of_birth_distinct_emails_4w"]

# 5 câu hỏi dùng chung cho mọi nguồn dữ liệu (BAF, SQLi, prompt injection...)
QUESTIONS = {
    "decision": {"type": "choice",
                 "instructions": "First-pass security verdict for this request.",
                 "criteria": {
                     "BLOCK": "Clear malicious signals; reject now.",
                     "ESCALATE_HUMAN": "High risk or high impact; hold for admin review.",
                     "TRIGGER_AI": "Suspicious but inconclusive; investigate further.",
                     "ALLOW": "No meaningful risk; let it through."}},
    "trace_group": {"type": "choice",
                    "instructions": "If investigated, which agent's evidence matters most?",
                    "criteria": {
                        "audit_log": "History, velocity and identity records in logs/DB.",
                        "internet_cti": "Device, network, session and geo signals.",
                        "syntax": "Structure of code, SQL or prompt payload.",
                        "none": "No investigation lead."}},
    "needs_investigation": {"type": "noul",
                            "instructions": "Is the request alone insufficient for a confident verdict?",
                            "criteria": {"false": "The request alone is enough to decide.",
                                         "true": "More evidence is needed before deciding."}},
    "risk": {"type": "score", "instructions": "How risky is this request?",
             "criteria": ["none", "low", "elevated", "high"]},
    "urgency": {"type": "score", "instructions": "How urgently must security act?",
                "criteria": ["routine", "soon", "prompt", "immediate"]},
}
QUESTIONS_STR = jdump(QUESTIONS)

## 3. Hàm làm sạch và rút gọn (số → nhãn chữ)

In [ ]:
def clean(df):
    n0 = len(df)
    df = df.drop_duplicates().copy()
    for c in NEG_MISSING:
        if c in df:
            df.loc[df[c] < 0, c] = np.nan
    const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    df = df.drop(columns=const).reset_index(drop=True)
    df.insert(0, "id", [f"baf_{i:07d}" for i in range(len(df))])
    log(f"Làm sạch: {n0} -> {len(df)} dòng; bỏ cột hằng: {const}")
    return df

def qedges(s, qs):
    return np.unique(s.dropna().quantile(qs).values)

def bucket(s, edges, labels, missing="unknown"):
    idx = np.clip(np.digitize(s.values, edges), 0, len(labels) - 1)
    out = np.array(labels, dtype=object)[idx]
    out[s.isna().values] = missing
    return out

def fit_reference(df, mask):
    """Ngưỡng phân vị chỉ tính trên các tháng train -> không rò rỉ sang valid/test."""
    r = df[mask]
    return {"vel_q": {c: qedges(r[c], [0.5, 0.8, 0.95]) for c in VEL_COLS},
            "p95": {c: r[c].quantile(0.95) for c in VEL_COLS},
            "sess_q": qedges(r["session_length_in_minutes"], [0.05, 0.5, 0.95]),
            "sess_p05": r["session_length_in_minutes"].quantile(0.05),
            "credit_q": qedges(r["credit_risk_score"], [0.25, 0.75, 0.95]),
            "limit_q": qedges(r["proposed_credit_limit"], [0.5, 0.9]),
            "limit_p90": r["proposed_credit_limit"].quantile(0.90),
            "bal_q": qedges(r["intended_balcon_amount"], [0.5, 0.9]),
            "days_q": qedges(r["days_since_request"], [0.5, 0.95])}

def shorten(df, ref):
    b = pd.DataFrame(index=df.index)
    vel = ["low", "normal", "high", "very_high"]
    for c, e in ref["vel_q"].items():
        b[c] = bucket(df[c], e, vel[: len(e) + 1])
    months = ["under_6m", "6m_2y", "2y_10y", "over_10y"]
    for c in ["prev_address_months_count", "current_address_months_count", "bank_months_count"]:
        b[c] = bucket(df[c], np.array([6, 24, 120.]), months)
    b["name_email_similarity"] = bucket(df["name_email_similarity"], np.array([.2, .5, .8]),
                                        ["very_low", "low", "medium", "high"])
    b["session_length_in_minutes"] = bucket(df["session_length_in_minutes"], ref["sess_q"],
                                            ["very_short", "short", "normal", "very_long"])
    b["credit_risk_score"] = bucket(df["credit_risk_score"], ref["credit_q"],
                                    ["low", "medium", "high", "very_high"])
    b["proposed_credit_limit"] = bucket(df["proposed_credit_limit"], ref["limit_q"],
                                        ["small", "medium", "large"])
    b["intended_balcon_amount"] = bucket(df["intended_balcon_amount"], ref["bal_q"],
                                         ["small", "medium", "large"])
    b["days_since_request"] = bucket(df["days_since_request"], ref["days_q"],
                                     ["recent", "normal", "stale"])
    b["income"] = [f"decile_{int(round(v * 10))}" for v in df["income"]]
    d = df["device_distinct_emails_8w"]
    b["device_distinct_emails_8w"] = np.where(
        d.isna(), "unknown", np.where(d >= 2, "2+", d.fillna(0).astype(int).astype(str)))
    for c in ["email_is_free", "phone_home_valid", "phone_mobile_valid",
              "has_other_cards", "foreign_request", "keep_alive_session"]:
        b[c] = df[c].astype(bool)
    for c in ["payment_type", "employment_status", "housing_status", "source", "device_os"]:
        b[c] = df[c].astype(str).str.lower()
    return b

def build_state(r):
    return {
        "workflow": WORKFLOW,
        "request": {"action": "open_bank_account", "channel": r["source"],
                    "payment_type": r["payment_type"], "credit_limit": r["proposed_credit_limit"],
                    "initial_deposit": r["intended_balcon_amount"], "foreign": r["foreign_request"]},
        "identity": {"name_email_match": r["name_email_similarity"], "free_email": r["email_is_free"],
                     "home_phone_valid": r["phone_home_valid"],
                     "mobile_phone_valid": r["phone_mobile_valid"],
                     "emails_same_dob_4w": r["date_of_birth_distinct_emails_4w"]},
        "history": {"current_address": r["current_address_months_count"],
                    "previous_address": r["prev_address_months_count"],
                    "bank_tenure": r["bank_months_count"], "other_cards": r["has_other_cards"],
                    "employment": r["employment_status"], "housing": r["housing_status"]},
        "velocity": {"apps_6h": r["velocity_6h"], "apps_24h": r["velocity_24h"],
                     "apps_4w": r["velocity_4w"], "apps_same_zip_4w": r["zip_count_4w"],
                     "apps_same_branch_8w": r["bank_branch_count_8w"]},
        "device": {"os": r["device_os"], "emails_on_device_8w": r["device_distinct_emails_8w"],
                   "session_length": r["session_length_in_minutes"],
                   "keep_alive": r["keep_alive_session"]},
        "financial": {"income": r["income"], "credit_risk": r["credit_risk_score"]},
    }

## 4. Red flags, teacher model và gold mềm

In [ ]:
def red_flags(df, ref):
    """Điểm bằng chứng theo nhóm truy vết: audit_log (lịch sử/tần suất) và internet_cti (thiết bị/phiên)."""
    p = ref["p95"]
    audit = (1.0 * (df["velocity_6h"] >= p["velocity_6h"])
             + 0.5 * (df["velocity_24h"] >= p["velocity_24h"])
             + 1.0 * (df["date_of_birth_distinct_emails_4w"] >= p["date_of_birth_distinct_emails_4w"])
             + 0.5 * (df["zip_count_4w"] >= p["zip_count_4w"])
             + 0.5 * ((df["current_address_months_count"] < 6) | df["current_address_months_count"].isna())
             + 1.0 * (df["name_email_similarity"] < 0.2)
             + 0.25 * (df["email_is_free"] == 1)
             + 0.5 * ((df["phone_home_valid"] == 0) & (df["phone_mobile_valid"] == 0)))
    cti = (1.0 * (df["device_distinct_emails_8w"] >= 2)
           + 1.0 * (df["foreign_request"] == 1)
           + 0.5 * (df["keep_alive_session"] == 0)
           + 0.5 * (df["session_length_in_minutes"] < ref["sess_p05"]))
    burst = (df["velocity_6h"] >= p["velocity_6h"]).values
    exposure = (df["proposed_credit_limit"] >= ref["limit_p90"]).values
    return audit.values.astype(float), cti.values.astype(float), burst, exposure

def teacher_oof(df, folds):
    """Xác suất gian lận out-of-fold: mỗi dòng được chấm bởi model KHÔNG học dòng đó."""
    drop = ["id", "fraud_bool", "month"] + [c for c in PROTECTED_COLS if c in df]
    X = pd.get_dummies(df.drop(columns=drop)).astype(np.float32)
    y = df["fraud_bool"].values
    p = np.zeros(len(df), dtype=np.float32)
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    for k, (tr, va) in enumerate(skf.split(X, y)):
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, random_state=SEED)
        m.fit(X.iloc[tr], y[tr])
        p[va] = m.predict_proba(X.iloc[va])[:, 1]
        log(f"Teacher fold {k + 1}/{folds} xong")
    log(f"Teacher AUC (out-of-fold) = {roc_auc_score(y, p):.4f}  (BAF thật thường ~0.88-0.90)")
    return p

def soft_labels(teacher_p, base_rate, fraud, audit, cti, burst, exposure, k, anchor,
                noise=0.4, protect_legit=True):
    """K annotator có ngưỡng nhiễu bỏ phiếu -> tỉ lệ phiếu = xác suất trong gold."""
    rng = np.random.default_rng(SEED)
    n = len(teacher_p)
    t_sig = np.clip(np.log10(np.clip(teacher_p, 1e-6, 1) / base_rate) / np.log10(30), -1, 1)[:, None]
    latent = (rng.uniform(1.5, 2.5, k) * t_sig + rng.uniform(.4, .7, k) * (audit + cti)[:, None]
              + rng.uniform(.6, 1.2, k) * anchor * fraud[:, None] + rng.normal(0, noise, (n, k)))
    lvl = ((latent >= rng.uniform(.5, 1.0, k)).astype(int) + (latent >= rng.uniform(1.3, 1.9, k))
           + (latent >= rng.uniform(2.1, 2.8, k)))
    ev = np.maximum(audit, cti)[:, None] + rng.normal(0, .3, (n, k))
    e_weak, e_strong = rng.uniform(.4, .8, k), rng.uniform(1.2, 1.8, k)
    exp_k = exposure[:, None] & (rng.random((n, k)) < .9)
    # thứ tự chỉ số theo DECISIONS: 0 BLOCK, 1 ESCALATE_HUMAN, 2 TRIGGER_AI, 3 ALLOW
    dec = np.select([lvl == 0, (lvl == 1) & (ev < e_weak), lvl == 1,
                     (lvl == 2) & exp_k, lvl == 2, (lvl == 3) & (ev >= e_strong)],
                    [3, 3, 2, 1, 2, 0], default=1)
    if protect_legit:   # khách hợp lệ: không bao giờ BLOCK, tối đa là ESCALATE_HUMAN
        dec = np.where((dec == 0) & (fraud[:, None] == 0), 1, dec)
    # có rủi ro thật nhưng bằng chứng trong request chưa đủ mạnh -> cần điều tra thêm
    need = (lvl >= 1) & (ev < e_strong)
    a_j = audit[:, None] + rng.normal(0, .3, (n, k))
    c_j = cti[:, None] + rng.normal(0, .3, (n, k))
    trace = np.where(np.maximum(a_j, c_j) < rng.uniform(.4, .7, k), 3, np.where(a_j >= c_j, 0, 1))
    urg = np.clip(lvl + (exp_k & (lvl >= 1)) + (burst[:, None] & (lvl >= 2)), 0, 3)
    frac = lambda m, c: np.stack([(m == i).mean(1) for i in range(c)], 1)
    return {"decision": frac(dec, 4), "trace_group": frac(trace, 4),
            "needs_investigation": need.mean(1),
            "risk": frac(lvl, 4), "urgency": frac(urg, 4)}

def conf(p):
    kk = len(p)
    return float(round((max(p) - 1 / kk) / (1 - 1 / kk), 6))

def gold_row(i, sl):
    r6 = lambda x: float(round(float(x), 6))
    d, t, pt = sl["decision"][i], sl["trace_group"][i], float(sl["needs_investigation"][i])
    g = {"decision": {"type": "choice", "label": DECISIONS[int(d.argmax())], "confidence": conf(d),
                      "probabilities": {a: r6(b) for a, b in zip(DECISIONS, d)}},
         "trace_group": {"type": "choice", "label": TRACE_OPTS[int(t.argmax())], "confidence": conf(t),
                         "probabilities": {a: r6(b) for a, b in zip(TRACE_OPTS, t)}},
         "needs_investigation": {"type": "noul", "label": "true" if pt >= .5 else "false",
                                 "noul": r6(pt), "confidence": conf([pt, 1 - pt]),
                                 "probabilities": {"false": r6(1 - pt), "true": r6(pt)}}}
    for q in ["risk", "urgency"]:
        p = sl[q][i]
        g[q] = {"type": "score", "label": str(int(p.argmax())),
                "score": r6((p * np.arange(4)).sum()), "confidence": conf(p),
                "probabilities": {str(j): r6(v) for j, v in enumerate(p)}}
    return g

def sample_and_split(df):
    """train: tháng < VALID_MONTH, cân bằng lại | valid, test: tỉ lệ tự nhiên."""
    pool = df[df["month"] < VALID_MONTH]
    pos, neg = pool[pool.fraud_bool == 1], pool[pool.fraud_bool == 0]
    n_neg = min(len(neg), NEG_RATIO * len(pos))
    hard = neg.nlargest(int(n_neg * HARD_FRAC), "teacher_p")
    rand = neg.drop(hard.index).sample(n=n_neg - len(hard), random_state=SEED)
    train = pd.concat([pos, hard, rand]).assign(split="train")
    va, te = df[df["month"] == VALID_MONTH], df[df["month"] > VALID_MONTH]
    valid = va.sample(n=min(VALID_N, len(va)), random_state=SEED).assign(split="valid")
    test = te.sample(n=min(TEST_N, len(te)), random_state=SEED).assign(split="test")
    return pd.concat([train, valid, test]).sample(frac=1, random_state=SEED).reset_index(drop=True)

## 5. Chạy: đọc và làm sạch

In [ ]:
raw = pd.read_csv(CSV_PATH, nrows=NROWS)
log(f"Đọc {len(raw):,} dòng, {raw.shape[1]} cột")
df = clean(raw); del raw
BASE_RATE = df["fraud_bool"].mean()
log(f"Tỉ lệ fraud = {BASE_RATE:.4%}  ({int(df.fraud_bool.sum()):,} ca)")
df.head(3)

## 6. Teacher model (bước lâu nhất)

In [ ]:
df["teacher_p"] = teacher_oof(df, TEACHER_FOLDS)
REF = fit_reference(df, df["month"] < VALID_MONTH)

## 7. Lấy mẫu, chia tập, tạo gold mềm

In [ ]:
data = sample_and_split(df)
audit, cti, burst, exposure = red_flags(data, REF)
SL = soft_labels(data["teacher_p"].values, BASE_RATE, data["fraud_bool"].values,
                 audit, cti, burst, exposure, N_ANNOTATORS, LABEL_ANCHOR,
                 noise=NOISE, protect_legit=PROTECT_LEGIT)
data["decision"] = np.array(DECISIONS)[SL["decision"].argmax(1)]
short = shorten(data, REF)
data.groupby("split").agg(n=("id", "size"), fraud_rate=("fraud_bool", "mean"),
                          months=("month", lambda s: sorted(s.unique())))

## 8. Ghi file: train / valid / test (+ file gộp)

In [ ]:
files = {s: open(f"{OUT_DIR}/{s}.jsonl", "w", encoding="utf-8") for s in ["train", "valid", "test"]}
f_all = open(f"{OUT_DIR}/baf_all.jsonl", "w", encoding="utf-8") if WRITE_ALL else None
lens = []
for i, (row, split) in enumerate(zip(short.to_dict("records"), data["split"])):
    st = jdump(build_state(row))
    lens.append(len(st))
    line = jdump({"state": st, "questions": QUESTIONS_STR, "gold": jdump(gold_row(i, SL))}) + "\n"
    files[split].write(line)
    if f_all:
        f_all.write(line)
for f in list(files.values()) + ([f_all] if f_all else []):
    f.close()
data[["id", "month", "fraud_bool", "teacher_p", "split", "decision"]].to_csv(
    f"{OUT_DIR}/baf_meta.csv", index=False)
log(f"Đã ghi {len(data):,} dòng. State TB {np.mean(lens):.0f} ký tự (max {max(lens)})")

## 9. Kiểm tra chất lượng

In [ ]:
from sklearn.metrics import roc_auc_score
P = SL["decision"]
need = SL["needs_investigation"]
tr = data["split"] == "train"
checks = {
    "Teacher AUC trên test (kỳ vọng 0.88-0.90)":
        round(roc_auc_score(data.loc[data.split == "test", "fraud_bool"],
                            data.loc[data.split == "test", "teacher_p"]), 4),
    "Số khách hợp lệ bị gán BLOCK (phải = 0 khi PROTECT_LEGIT)":
        int(((data["decision"] == "BLOCK") & (data["fraud_bool"] == 0)).sum()),
    "Tỉ lệ fraud bị gán ALLOW ở train (nên < 25%)":
        round(float(((data["decision"] == "ALLOW") & (data["fraud_bool"] == 1))[tr].sum()
                    / data.loc[tr, "fraud_bool"].sum()), 3),
    "Tương quan needs_investigation với P(ESC)+P(TRIGGER) (phải < 0.95)":
        round(float(np.corrcoef(need, P[:, 1] + P[:, 2])[0, 1]), 3),
    "Tỉ lệ dòng mọi annotator đồng ý tuyệt đối":
        round(float((P.max(1) == 1).mean()), 3),
}
report = "\n".join([
    "=== REX BOT / BAF REPORT ===",
    f"Nguồn: {CSV_PATH}", f"Tỉ lệ fraud gốc: {BASE_RATE:.4%}", "",
    "Số dòng theo tập:",
    data.groupby("split").agg(n=("id", "size"), fraud_rate=("fraud_bool", "mean")).round(4).to_string(), "",
    "Tỉ lệ nhãn decision (argmax) theo tập (%):",
    (pd.crosstab(data["split"], data["decision"], normalize="index") * 100).round(1).to_string(), "",
    "Decision theo fraud_bool (train):",
    pd.crosstab(data.loc[tr, "decision"], data.loc[tr, "fraud_bool"]).to_string(), "",
    "Decision theo fraud_bool (test, tỉ lệ tự nhiên):",
    pd.crosstab(data.loc[data.split == "test", "decision"],
                data.loc[data.split == "test", "fraud_bool"]).to_string(), "",
    "KIỂM TRA TỰ ĐỘNG:",
    *[f"  - {k}: {v}" for k, v in checks.items()], "",
    f"Độ dài state: TB {np.mean(lens):.0f}, max {max(lens)} ký tự; questions: {len(QUESTIONS_STR)} ký tự",
])
open(f"{OUT_DIR}/report.txt", "w", encoding="utf-8").write(report)
print(report)

In [ ]:
# Xem 1 mẫu TRIGGER_AI trong tập train
ex = json.loads(open(f"{OUT_DIR}/train.jsonl").readline())
for line in open(f"{OUT_DIR}/train.jsonl"):
    r = json.loads(line)
    if json.loads(r["gold"])["decision"]["label"] == "TRIGGER_AI":
        ex = r; break
print("STATE:\n", json.dumps(json.loads(ex["state"]), indent=1, ensure_ascii=False))
print("\nGOLD.decision:\n", json.dumps(json.loads(ex["gold"])["decision"], indent=1))

## 10. Đóng gói để tải về

In [ ]:
zip_path = "/kaggle/working/rexbot_baf.zip" if OUT_DIR.startswith("/kaggle") else OUT_DIR + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, f), arcname=f"rexbot_baf/{f}")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"{f:20s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:8.1f} MB")
print(f"\nFile nén: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

## Dùng tiếp ở notebook train

Sau khi **Save Version** (Save & Run All), tab **Output** có thư mục `rexbot_baf/`.
Ở notebook fine-tune, **Add Input → Your Work → notebook này**, rồi đọc:

```python
from datasets import load_dataset
ds = load_dataset("json", data_files={
    "train": "/kaggle/input/<tên-notebook>/rexbot_baf/train.jsonl",
    "validation": "/kaggle/input/<tên-notebook>/rexbot_baf/valid.jsonl",
    "test": "/kaggle/input/<tên-notebook>/rexbot_baf/test.jsonl"})
```

Ba cột `state`, `questions`, `gold` đều là **chuỗi JSON** (giống `LocalLLaMA/typed-decisions`) → dùng `json.loads` khi cần.